In [1]:
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('.setup_complete'):
    # Install xvfb and our launcher script for it
    !apt-get install -y xvfb
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/xvfb -O ../xvfb

    # Download dependencies from Github
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/atari_wrappers.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/env_batch.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/runners.py

    # Update the gym environment to be compatible with the Atari environment
    !pip install -q gymnasium[atari,accept-rom-license]
    !pip install -q tensorboardX

    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ['DISPLAY'] = ':1'

# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.



In [2]:
import numpy as np
import gymnasium as gym
from atari_wrappers import nature_dqn_env


env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = 16  # change this if you have more than 8 CPU ;)
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (vers

Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class NatureDQN_A2C(nn.Module):
    def __init__(self, num_actions):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels=4, out_channels=32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1)
        
        self.fc = nn.Linear(in_features=64 * 7 * 7, out_features=512)
        
        self.logits = nn.Linear(in_features=512, out_features=num_actions)
        
        self.state_value = nn.Linear(in_features=512, out_features=1)
        
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
                    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        
        x = F.relu(self.fc(x.reshape(x.size(0), -1)))
        
        logits = self.logits(x)
        values = self.state_value(x).squeeze(-1)
        
        return logits, values

num_actions = env.action_space.n
model = NatureDQN_A2C(num_actions).to(device)

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [4]:
class Policy:
    def __init__(self, model):
        self.model = model

    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ['actions', 'logits', 'log_probs', 'values'].

        inputs_t = torch.tensor(np.array(inputs), dtype=torch.float32, device=device)
        
        logits, values = self.model(inputs_t)
        
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)
        
        dist = torch.distributions.Categorical(probs)
        actions = dist.sample()
        
        return {
            "actions": actions.cpu().numpy(),
            "logits": logits,
            "log_probs": log_probs,
            "values": values
        }


Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [5]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* 'observations'
* 'rewards'
* 'resets'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it's different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t'=0}^{T - 1} \gamma^{t'}r_{t+t'} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory['resets']` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory['state']['latest_observation']`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [6]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    def __call__(self, trajectory):
        """Compute value targets for a given partial trajectory."""

        # This method should modify trajectory inplace by adding
        # an item with key 'value_targets' to it.

        rewards = np.array(trajectory['rewards']) 
        resets = np.array(trajectory['resets'])
        
        latest_obs = trajectory['state']['latest_observation']
        
        next_value = self.policy.act(latest_obs)['values'].detach().cpu().numpy()
        
        value_targets = []
        
        for t in reversed(range(len(rewards))):

            next_value = rewards[t] + self.gamma * next_value * (1.0 - resets[t])

            value_targets.append(next_value.copy())
            

        value_targets = list(reversed(value_targets))
        

        trajectory['value_targets'] = value_targets

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [7]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory):
        # Modify trajectory inplace.

        for key in list(trajectory.keys()):
            if key == 'state':
                continue
                
            val = trajectory[key]
            
            if isinstance(val, list):
                if len(val) > 0 and isinstance(val[0], torch.Tensor):
                    stacked = torch.stack(val)
                    trajectory[key] = stacked.view(-1, *stacked.shape[2:])
                else:
                    arr = np.array(val)
                    trajectory[key] = arr.reshape(-1, *arr.shape[2:])


In [8]:
model = NatureDQN_A2C(env.action_space.n).to(device)
policy = Policy(model)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)


Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

In [9]:
class A2C:
    def __init__(self,
                 policy,
                 optimizer,
                 value_loss_coef=0.25,
                 entropy_coef=0.01,
                 max_grad_norm=0.5):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

    def policy_loss(self, trajectory):
        # You will need to compute advantages here.
        states = trajectory['observations']
        actions = torch.tensor(trajectory['actions'], dtype=torch.long, device=device)
        value_targets = torch.tensor(trajectory['value_targets'], dtype=torch.float32, device=device)
        
        result = self.policy.act(states)
        log_probs = result['log_probs']
        values = result['values']
        
        advantages = (value_targets - values).detach()
        log_probs_for_actions = log_probs[range(len(actions)), actions]
        
        return -torch.mean(log_probs_for_actions * advantages)

    def value_loss(self, trajectory):
        states = trajectory['observations']
        value_targets = torch.tensor(trajectory['value_targets'], dtype=torch.float32, device=device)
        
        result = self.policy.act(states)
        values = result['values']
        
        return torch.mean((values - value_targets) ** 2)

    def loss(self, trajectory):
        states = trajectory['observations']
        actions = torch.tensor(trajectory['actions'], dtype=torch.long, device=device)
        value_targets = torch.tensor(trajectory['value_targets'], dtype=torch.float32, device=device)
        
        result = self.policy.act(states)
        logits = result['logits']
        log_probs = result['log_probs']
        values = result['values']
        
        advantages = (value_targets - values).detach()
        log_probs_for_actions = log_probs[range(len(actions)), actions]
        policy_loss_val = -torch.mean(log_probs_for_actions * advantages)
        
        value_loss_val = torch.mean((values - value_targets) ** 2)
        
        probs = F.softmax(logits, dim=-1)
        entropy_val = -torch.sum(probs * log_probs, dim=-1).mean()
        
        total_loss = policy_loss_val + self.value_loss_coef * value_loss_val - self.entropy_coef * entropy_val
        return total_loss

    def step(self, trajectory):
        self.optimizer.zero_grad()
        loss_val = self.loss(trajectory)
        loss_val.backward()
        
        torch.nn.utils.clip_grad_norm_(self.policy.model.parameters(), self.max_grad_norm)
        
        self.optimizer.step()

Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.

In [13]:
#if you use TensorboardSummaries
%load_ext tensorboard
%tensorboard --logdir logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 200582), started 0:59:14 ago. (Use '!kill 200582' to kill it.)

In [11]:
from tqdm import trange

optimizer = torch.optim.RMSprop(model.parameters(), lr=7e-4, alpha=0.99, eps=1e-5)

a2c = A2C(policy, optimizer, value_loss_coef=0.5, entropy_coef=0.01, max_grad_norm=0.5)

# total_iterations = 250000
total_iterations = 125000
initial_lr = 7e-4

tbar = trange(total_iterations)
for i in tbar:
    lr = initial_lr * (1.0 - i / total_iterations)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
        
    trajectory = runner.get_next()
    
    a2c.step(trajectory)
    
    if i % 1000 == 0:
        tbar.set_description(f"LR: {lr:.5f}")

LR: 0.00001: 100%|██████████| 125000/125000 [56:57<00:00, 36.58it/s]


In [12]:
torch.save(model.state_dict(), "a2c_space_invaders.pt")

![График обучения A2C](plots/plots.png)

### Небольшой вывод

В ходе выполнения лабораторной работы был реализован и обучен алгоритм Advantage Actor-Critic (A2C) для среды Space Invaders на Atari. 

Для стабилизации процесса обучения и снижения дисперсии градиентов количество параллельных сред было установлено n_envs = 16, а коэффициент лосса ценности (value_loss_coef) - 0.5. 

В результате обучения на протяжении 10 млн шагов (выполненных за 125 000 итераций):
1. Средняя награда за последние 100 эпизодов (reward_mean_100) успешно преодолела целевой порог в 610 очков.
2. Максимальная награда за отдельную игру достигла пикового значения в 2405 очков.

Графики обучения показывают плавную и стабильную сходимость без признаков расхождения политики, что полностью подтверждает корректность математической реализации всех компонентов алгоритма (Actor, Critic, расчет преимуществ и лоссов).

В алгоритме A2C, в отличие от стандартного DQN, архитектура разделена на Актера (выбор действий) и Критика (оценка состояний), а для снижения дисперсии при обучении используется функция преимущества (Advantage). Вместо Replay Buffer применяется параллельное выполнение сред, что стабилизирует процесс и ускоряет вычисления. Подход Actor-Critic обеспечивает более плавное изменение стратегии и компенсирует ошибки оценки ценности состояний.

### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.